# Session 2.3 : Relational Data

_Analytics Through Coding Autumn 2026_

---

It is rare that an analysis uses only one table.

Information is often split across several related datasets:

* one table contains the observations we want to analyse;
* other tables contain additional information about those observations.

The analytical task is not simply to **join tables**. We also need to check that the join produced the result we expected.

In this session we will focus on:

* identifying the variables that link tables;
* understanding one-to-many relationships;
* combining tables with `merge()`;
* checking whether joins changed the number of observations;
* identifying records that did not match;
* using semi-join and anti-join logic;
* combining several related datasets.

---

## Starting out

We will continue using the `nycflights13` datasets.

The main `flights` table records individual flights. Other tables contain information about airlines, airports, aircraft and weather.

In [1]:
import pandas as pd
import numpy as np

flights = pd.read_csv("../Data/nycflights13_flights.csv", index_col=0)
flights.reset_index(drop=True, inplace=True)

airlines = pd.read_csv("../Data/nycflights13_airlines.csv")
airports = pd.read_csv("../Data/nycflights13_airports.csv")
planes = pd.read_csv("../Data/nycflights13_planes.csv")
weather = pd.read_csv("../Data/nycflights13_weather.csv")

print("flights:", flights.shape)
print("airlines:", airlines.shape)
print("airports:", airports.shape)
print("planes:", planes.shape)
print("weather:", weather.shape)

flights: (202066, 19)
airlines: (16, 2)
airports: (1458, 8)
planes: (3322, 9)
weather: (26115, 15)


## How are the tables related?

![GitHub Codespaces](fligths_data.png)

Before joining anything, identify:

1. **What does one row represent in each table?**
2. **Which variable links the tables?**
3. **Is the linking variable unique in one of the tables?**

For example:

* one row in `flights` = one flight;
* one row in `airlines` = one airline;
* `carrier` links the two tables.

Many flights can belong to the same airline, so this is a **many-to-one** relationship from `flights` to `airlines`.



In [2]:
display(flights[["carrier", "flight", "origin", "dest"]].head())
display(airlines.head())

,carrier,flight,origin,dest
0,UA,1471,EWR,RSW
1,DL,1765,JFK,SFO
2,EV,4129,EWR,DCA
3,EV,5796,EWR,CLT
4,B6,939,JFK,BQN


,carrier,name
0,9E,Endeavor Air Inc.
1,AA,American Airlines Inc.
2,AS,Alaska Airlines Inc.
3,B6,JetBlue Airways
4,DL,Delta Air Lines Inc.


A useful check is whether the lookup key is unique in the table that is supposed to contain one row per entity.

In [4]:
airlines['carrier'].is_unique

True

<div class="alert alert-warning">
<b>Why does uniqueness matter?</b>

If we expect one airline record for each carrier but the airline table contains duplicate carrier codes, a join could unexpectedly duplicate rows in the flights table.

Always understand the relationship between the join keys before merging.
</div>

## Question 1: What is the full airline name for each flight?

The `flights` table only contains short carrier codes such as `UA`, `DL` and `B6`.

The `airlines` table contains the corresponding airline names.

For this question, we want to keep **every flight** and add airline information where a match exists.

That makes a **left join** the natural choice.

In [8]:
flights_with_name = flights.merge(
    airlines,
    on='carrier',
    how='left',
    validate='many_to_one'
)

flights_with_name

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,United Air Lines Inc.
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,Delta Air Lines Inc.
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,ExpressJet Airlines Inc.
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,ExpressJet Airlines Inc.
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,JetBlue Airways
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202061,2013,1,27,1311.0,1315,-4.0,1451.0,1504,-13.0,US,1895,N192UW,EWR,CLT,77.0,529,13,15,2013-01-27 13:00:00,US Airways Inc.
202062,2013,8,8,2145.0,1800,225.0,8.0,2039,209.0,DL,61,N685DA,LGA,ATL,104.0,762,18,0,2013-08-08 18:00:00,Delta Air Lines Inc.
202063,2013,1,30,2248.0,2135,73.0,149.0,36,73.0,B6,11,N809JB,JFK,FLL,166.0,1069,21,35,2013-01-30 21:00:00,JetBlue Airways
202064,2013,3,18,957.0,1000,-3.0,1242.0,1234,8.0,DL,1847,N397DA,LGA,ATL,112.0,762,10,0,2013-03-18 10:00:00,Delta Air Lines Inc.


The argument:

```python
validate="many_to_one"
```

asks pandas to confirm that many rows in `flights` are allowed to match one row in `airlines`.

If the relationship is not what we expected, pandas will raise an error rather than silently producing a potentially incorrect dataset.

## Validate the join

A join is not finished simply because the code ran.

Before the join:

> one row = one flight

After adding airline information:

> one row should still = one flight

Therefore the number of rows should not change.

In [10]:
print(len(flights))
print(len(flights_with_name))

202066
202066


We should also check whether any flights failed to match an airline.

In [12]:
print(flights_with_name['name'].isna().sum())

0


### Exercise 1 — Join and validate

Join `flights` to `airlines` using a left join.

Store the result in `flight_airline_check`.

Then:

1. Use `validate="many_to_one"`.
2. Confirm that the number of rows has not changed.
3. Count the number of missing airline names.
4. Display `carrier`, `name`, `origin` and `dest` for the first 10 rows.

In [14]:
display(flights.head(5))
display(airlines.head(5))


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00


,carrier,name
0,9E,Endeavor Air Inc.
1,AA,American Airlines Inc.
2,AS,Alaska Airlines Inc.
3,B6,JetBlue Airways
4,DL,Delta Air Lines Inc.


## Inner, left, right and outer joins

Pandas supports several join types.

| Join | What it keeps |
|---|---|
| `inner` | Only rows with a match in both tables |
| `left` | All rows from the left table, plus matching information from the right |
| `right` | All rows from the right table, plus matching information from the left |
| `outer` | All rows from both tables |

![GitHub Codespaces](mutating_joins.png)

For analytical work, the important question is not:

> Which join type do I know?

It is:

> **Which observations should remain in the resulting dataset?**

For example, compare an inner and left join between flights and airlines.

In [16]:
inner = flights.merge(
    airlines,
    on='carrier',
    how='inner'
)

left = flights.merge(
    airlines,
    on='carrier',
    how='inner'
)

display(inner.describe())
display(left.describe())

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,flight,air_time,distance,hour,minute
count,202066.0,202066.000000,202066.000000,197177.000000,202066.000000,197177.000000,196899.000000,202066.000000,196448.000000,202066.000000,196448.000000,202066.000000,202066.000000,202066.000000
mean,2013.0,6.555225,15.704255,1348.842781,1343.998822,12.623232,1501.047867,1535.118382,6.944759,1971.788213,150.644170,1039.875961,13.177635,26.235319
std,0.0,3.411528,8.770402,487.940379,467.023647,40.287873,533.225113,497.554216,44.741547,1632.396340,93.688056,733.506709,4.658215,19.301681
min,2013.0,1.000000,1.000000,1.000000,106.000000,-33.000000,1.000000,1.000000,-79.000000,1.000000,20.000000,17.000000,1.000000,0.000000
25%,2013.0,4.000000,8.000000,907.000000,906.000000,-5.000000,1104.000000,1124.000000,-17.000000,559.000000,82.000000,502.000000,9.000000,8.000000
50%,2013.0,7.000000,16.000000,1400.000000,1359.000000,-2.000000,1535.000000,1555.000000,-5.000000,1496.000000,129.000000,872.000000,13.000000,29.000000
75%,2013.0,10.000000,23.000000,1743.000000,1729.000000,11.000000,1939.000000,1944.000000,14.000000,3461.000000,192.000000,1389.000000,17.000000,44.000000
max,2013.0,12.000000,31.000000,2400.000000,2359.000000,1301.000000,2400.000000,2359.000000,1272.000000,6181.000000,695.000000,4983.000000,23.000000,59.000000


,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,flight,air_time,distance,hour,minute
count,202066.0,202066.000000,202066.000000,197177.000000,202066.000000,197177.000000,196899.000000,202066.000000,196448.000000,202066.000000,196448.000000,202066.000000,202066.000000,202066.000000
mean,2013.0,6.555225,15.704255,1348.842781,1343.998822,12.623232,1501.047867,1535.118382,6.944759,1971.788213,150.644170,1039.875961,13.177635,26.235319
std,0.0,3.411528,8.770402,487.940379,467.023647,40.287873,533.225113,497.554216,44.741547,1632.396340,93.688056,733.506709,4.658215,19.301681
min,2013.0,1.000000,1.000000,1.000000,106.000000,-33.000000,1.000000,1.000000,-79.000000,1.000000,20.000000,17.000000,1.000000,0.000000
25%,2013.0,4.000000,8.000000,907.000000,906.000000,-5.000000,1104.000000,1124.000000,-17.000000,559.000000,82.000000,502.000000,9.000000,8.000000
50%,2013.0,7.000000,16.000000,1400.000000,1359.000000,-2.000000,1535.000000,1555.000000,-5.000000,1496.000000,129.000000,872.000000,13.000000,29.000000
75%,2013.0,10.000000,23.000000,1743.000000,1729.000000,11.000000,1939.000000,1944.000000,14.000000,3461.000000,192.000000,1389.000000,17.000000,44.000000
max,2013.0,12.000000,31.000000,2400.000000,2359.000000,1301.000000,2400.000000,2359.000000,1272.000000,6181.000000,695.000000,4983.000000,23.000000,59.000000


If every carrier in `flights` is present in `airlines`, the inner and left joins will have the same number of rows.

If some carriers are missing from the lookup table, an inner join would silently remove those flights.

That is why validation matters.

## Finding unmatched records with `indicator=True`

Pandas can tell us whether each observation matched using the `indicator` argument.

In [20]:
inner = flights.merge(
    airlines,
    on='carrier',
    how='inner',
    indicator=True
)

inner.head(20)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name,_merge
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,...,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,United Air Lines Inc.,both
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,...,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,Delta Air Lines Inc.,both
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,...,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,ExpressJet Airlines Inc.,both
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,...,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,ExpressJet Airlines Inc.,both
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,...,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,JetBlue Airways,both
5,2013,11,14,2031.0,2025,6.0,2342.0,2340,2.0,UA,...,N518UA,JFK,LAX,342.0,2475,20,25,2013-11-14 20:00:00,United Air Lines Inc.,both
6,2013,3,30,1755.0,1755,0.0,2110.0,2114,-4.0,B6,...,N646JB,JFK,LAX,338.0,2475,17,55,2013-03-30 17:00:00,JetBlue Airways,both
7,2013,8,10,1544.0,1545,-1.0,2003.0,2001,2.0,DL,...,N3764D,JFK,SJU,205.0,1598,15,45,2013-08-10 15:00:00,Delta Air Lines Inc.,both
8,2013,3,27,1855.0,1900,-5.0,2019.0,2028,-9.0,9E,...,N8623A,JFK,BWI,37.0,184,19,0,2013-03-27 19:00:00,Endeavor Air Inc.,both
9,2013,5,13,1912.0,1915,-3.0,2055.0,2154,-59.0,DL,...,N3736C,JFK,PHX,262.0,2153,19,15,2013-05-13 19:00:00,Delta Air Lines Inc.,both


The `_merge` column contains:

* `both` — a match was found;
* `left_only` — the observation existed only in the left table;
* `right_only` — the observation existed only in the right table.

This is especially useful when we need to investigate failed matches.

## Question 2: Which destinations cannot be matched to the airport table?

The destination variable is called `dest` in `flights`, but the airport code is called `faa` in `airports`.

The variable names do not have to be identical.

We specify the matching columns explicitly.

In [38]:
display(flights.head(5))
display(airports.head(5))

dests_without_match = flights.merge(
    airports,
    left_on='dest',
    right_on='faa',
    how='left',
    validate='many_to_one',
    indicator=True
).query("_merge == 'left_only'")['dest'].unique()

display(dests_without_match)

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00
1,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00
2,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00
3,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00
4,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00


,faa,name,lat,lon,alt,tz,dst,tzone
0,04G,Lansdowne Airport,41.130472,-80.619583,1044,-5,A,America/New_York
1,06A,Moton Field Municipal Airport,32.460572,-85.680028,264,-6,A,America/Chicago
2,06C,Schaumburg Regional,41.989341,-88.101243,801,-6,A,America/Chicago
3,06N,Randall Airport,41.431912,-74.391561,523,-5,A,America/New_York
4,09J,Jekyll Island Airport,31.074472,-81.427778,11,-5,A,America/New_York


<StringArray>
['BQN', 'SJU', 'PSE', 'STT']
Length: 4, dtype: str

Now we can isolate destination codes that did not match an airport record.

This is a much more useful check than simply noticing missing airport names later.

A failed match could mean:

* the lookup table does not cover every destination;
* the join key uses a different coding convention;
* the source data contain an error;
* the wrong join key was selected.

Do not automatically treat an unmatched value as bad data.

### Exercise 2 — Airport join

Add destination airport information to the `flights` table.

1. Match `flights["dest"]` to `airports["faa"]`.
2. Use a left join.
3. Validate the relationship as `many_to_one`.
4. Use `indicator=True`.
5. Count how many flight records matched and did not match.
6. List the unique unmatched destination codes.

## Filtering joins: semi and anti join logic

Sometimes we do not want to add columns from another table.

Instead, we only want to know whether a match exists.

Two useful concepts are:

**Semi join**

> Keep rows from the left table where a match exists in the right table.

**Anti join**

> Keep rows from the left table where no match exists in the right table.

Pandas does not have dedicated `semi_join()` or `anti_join()` functions, but the logic is easy to implement.

### Semi join example

Keep only flights whose destination appears in the airport table.

### Anti join example

Keep only flights whose destination does **not** appear in the airport table.

The anti join is particularly useful for **data validation** because it identifies records that failed to find a match.

### Exercise 3 — Anti join

Using `flights` and `planes`, identify flights whose `tailnum` does not appear in the aircraft table.

1. Ignore flights where `tailnum` itself is missing.
2. Keep only unmatched flights.
3. Display the unique unmatched tail numbers.
4. Count how many flight records have an unmatched non-missing tail number.

## Question 3: Can we combine several lookup tables?

Yes. Joins can be chained.

For example, starting with one row per flight we can add:

* airline name using `carrier`;
* aircraft characteristics using `tailnum`;
* destination airport information using `dest` → `faa`.

The main rule is that we should validate each relationship rather than blindly joining tables together.

The number of columns increases because we have added information.

The number of rows should remain unchanged because one row should still represent one flight.

## Final Exercise — Build and validate an analytical dataset

Create a DataFrame called `flight_analysis` that contains:

* the original flight information;
* airline name;
* aircraft information;
* destination airport information.

Use left joins so that no flight is intentionally removed.

Then verify:

1. Each lookup relationship is treated as `many_to_one`.
2. The final number of rows equals the original number of flights.
3. Count missing airline names.
4. Count missing aircraft manufacturers.
5. Count missing destination airport names.

Finally display these columns for the first 10 rows:

* `carrier`
* airline `name`
* `tailnum`
* `manufacturer`
* `dest`
* destination airport name
* `arr_delay`

Because both airlines and airports contain a column called `name`, rename these clearly before joining.

## A join validation checklist

Before moving on from a merge, ask:

**1. What does one row represent before the join?**

**2. What should one row represent after the join?**

**3. Is the key unique where I expect it to be unique?**

**4. What relationship do I expect?**
* one-to-one
* one-to-many
* many-to-one
* many-to-many

**5. Did the number of rows change? Should it have?**

**6. Did any observations fail to match?**

**7. Did the join introduce unexpected missing values?**

A join that executes successfully is not necessarily a correct join.

## All Done!

In this session we worked with relational data and practised:

* identifying variables that connect datasets;
* recognising many-to-one relationships;
* combining datasets with `merge()`;
* choosing a join based on which observations should remain;
* using `validate=` to test the expected relationship;
* checking row counts after joining;
* using `indicator=True` to identify failed matches;
* implementing semi-join and anti-join logic;
* combining several lookup tables into one analytical dataset.

The key idea is:

> **Do not only ask whether the join ran. Ask whether the join produced the dataset you intended.**

We will now move from preparing and combining data to **exploratory analysis and visualisation**.